# Project 5: Indian Politics Sentiment Analysis

This project implements a sentiment analysis pipeline specialized for Indian political discourse.
We use a **Multilingual RoBERTa** model to handle the nuances of political language across different contexts.

## Pipeline Overview:
1. **Setup**: Install NLP libraries.
2. **Config**: Define model and dataset parameters.
3. **Data**: Load political text/tweets.
4. **Preprocessing**: Tokenize text for RoBERTa.
5. **Model**: Load the pre-trained classifier.
6. **Training**: Define trainer and evaluate performance.
7. **Inference**: Predict sentiment on custom political text.

In [ ]:
# Install necessary libraries for Hugging Face Transformers and Dataset handling
!pip install -q transformers datasets accelerate scikit-learn

In [ ]:
# Configuration parameters
# Using CardiffNLP's XLM-RoBERTa base model for multilingual sentiment
model_name = "cardiffnlp/twitter-xlm-roberta-base-sentiment"
dataset_name = "tweet_eval" # We'll use the sentiment subset of tweet_eval as a proxy
output_dir = "./sentiment_results"

# Training hyperparameters
batch_size = 16
learning_rate = 2e-5
epochs = 3

In [ ]:
from datasets import load_dataset

# Load the sentiment analysis dataset
print("Loading sentiment dataset...")
dataset = load_dataset(dataset_name, "sentiment")

# Display a sample from the training set to verify the structure
print(f"Sample data: {dataset['train'][0]}")

In [ ]:
from transformers import AutoTokenizer

# Initialize the tokenizer compatible with XLM-RoBERTa
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Tokenization function that pads and truncates text to a standard length
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True)

# Apply tokenization across the entire dataset in batches
tokenized_datasets = dataset.map(tokenize_function, batched=True)

In [ ]:
from transformers import AutoModelForSequenceClassification

# Load the pre-trained model with 3 output labels: Negative, Neutral, and Positive
print("Loading multilingual sentiment model...")
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3)

In [ ]:
from transformers import TrainingArguments, Trainer
import numpy as np
import evaluate

# Load the accuracy metric for evaluation
metric = evaluate.load("accuracy")

# Function to calculate metrics during training
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

# Define training arguments including evaluation strategy and learning rate
training_args = TrainingArguments(
    output_dir=output_dir,
    evaluation_strategy="epoch",
    learning_rate=learning_rate,
    per_device_train_batch_size=batch_size,
    num_train_epochs=epochs,
    weight_decay=0.01,
    report_to="none"
)

# Initialize the Trainer object for fine-tuning
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    compute_metrics=compute_metrics,
)

# To start training, uncomment the line below:
# trainer.train()

In [ ]:
import torch
from transformers import pipeline

# Initialize the sentiment analysis pipeline using our chosen model
sentiment_pipeline = pipeline("sentiment-analysis", model=model_name)

# Function to predict sentiment for custom Indian political text
def analyze_political_sentiment(text):
    result = sentiment_pipeline(text)
    # Mapping RoBERTa labels to human-readable sentiment categories
    label_map = {"LABEL_0": "Negative", "LABEL_1": "Neutral", "LABEL_2": "Positive"}
    return label_map.get(result[0]["label"], result[0]["label"])

# Example test case relating to Indian political/economic context
test_text = "Indian infrastructure growth is at an all-time high, driving massive economic progress."
print(f"Text Analysis: {test_text}")
print(f"Predicted Sentiment: {analyze_political_sentiment(test_text)}")